In [1]:
import duckdb

DATA_DIR = r"C:\Users\shyam\Downloads\synerise_dataset"

con = duckdb.connect("ecommerce_analytics.duckdb")

print("=" * 80)
print("CUSTOMER CART JOURNEY ANALYSIS")
print("=" * 80)

# ============================================================
# 1. MATCH CART ADDITIONS TO SUBSEQUENT EVENTS
# ============================================================

query = f"""
CREATE OR REPLACE TABLE cart_journey AS

WITH cart_adds AS (

    SELECT
        client_id,
        sku,
        CAST(timestamp AS TIMESTAMP) AS add_time
    FROM read_parquet('{DATA_DIR}/add_to_cart.parquet')
),

purchases AS (

    SELECT
        client_id,
        sku,
        CAST(timestamp AS TIMESTAMP) AS purchase_time
    FROM read_parquet('{DATA_DIR}/product_buy.parquet')
),

removals AS (

    SELECT
        client_id,
        sku,
        CAST(timestamp AS TIMESTAMP) AS remove_time
    FROM read_parquet('{DATA_DIR}/remove_from_cart.parquet')
),

next_purchase AS (

    SELECT
        a.client_id,
        a.sku,
        a.add_time,

        MIN(p.purchase_time) AS purchase_time

    FROM cart_adds a

    LEFT JOIN purchases p

        ON a.client_id = p.client_id
        AND a.sku = p.sku
        AND p.purchase_time > a.add_time

    GROUP BY
        a.client_id,
        a.sku,
        a.add_time
),

next_remove AS (

    SELECT
        a.client_id,
        a.sku,
        a.add_time,

        MIN(r.remove_time) AS remove_time

    FROM cart_adds a

    LEFT JOIN removals r

        ON a.client_id = r.client_id
        AND a.sku = r.sku
        AND r.remove_time > a.add_time

    GROUP BY
        a.client_id,
        a.sku,
        a.add_time
)

SELECT

    p.client_id,
    p.sku,
    p.add_time,

    p.purchase_time,
    r.remove_time,

    CASE

        WHEN p.purchase_time IS NULL
             AND r.remove_time IS NULL
            THEN 'NO_FOLLOWUP'

        WHEN p.purchase_time IS NOT NULL
             AND r.remove_time IS NULL
            THEN 'PURCHASED'

        WHEN p.purchase_time IS NULL
             AND r.remove_time IS NOT NULL
            THEN 'REMOVED'

        WHEN p.purchase_time < r.remove_time
            THEN 'PURCHASED'

        ELSE 'REMOVED'

    END AS journey_outcome,

    CASE

        WHEN p.purchase_time IS NOT NULL
             AND (
                 r.remove_time IS NULL
                 OR p.purchase_time < r.remove_time
             )
        THEN DATE_DIFF(
            'second',
            p.add_time,
            p.purchase_time
        )

        WHEN r.remove_time IS NOT NULL
        THEN DATE_DIFF(
            'second',
            r.add_time,
            r.remove_time
        )

        ELSE NULL

    END AS seconds_to_outcome

FROM next_purchase p

LEFT JOIN next_remove r

    ON p.client_id = r.client_id
    AND p.sku = r.sku
    AND p.add_time = r.add_time
"""

con.execute(query)

print("\nCart journey table created successfully!")


# ============================================================
# 2. OVERALL JOURNEY OUTCOMES
# ============================================================

print("\n" + "=" * 80)
print("CART JOURNEY OUTCOMES")
print("=" * 80)

result = con.execute("""
SELECT

    journey_outcome,

    COUNT(*) AS cart_events,

    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage

FROM cart_journey

GROUP BY journey_outcome

ORDER BY cart_events DESC
""").fetchdf()

print(result.to_string(index=False))


# ============================================================
# 3. TIME TO OUTCOME
# ============================================================

print("\n" + "=" * 80)
print("TIME TO CART OUTCOME")
print("=" * 80)

result = con.execute("""
SELECT

    journey_outcome,

    COUNT(*) AS events,

    ROUND(
        AVG(seconds_to_outcome) / 60,
        2
    ) AS avg_minutes,

    ROUND(
        MEDIAN(seconds_to_outcome) / 60,
        2
    ) AS median_minutes,

    ROUND(
        MIN(seconds_to_outcome) / 60,
        2
    ) AS min_minutes,

    ROUND(
        MAX(seconds_to_outcome) / 60,
        2
    ) AS max_minutes

FROM cart_journey

WHERE seconds_to_outcome IS NOT NULL

GROUP BY journey_outcome

ORDER BY events DESC
""").fetchdf()

print(result.to_string(index=False))


# ============================================================
# 4. PURCHASE VS REMOVE BY CATEGORY
# ============================================================

print("\n" + "=" * 80)
print("JOURNEY OUTCOME BY CATEGORY")
print("=" * 80)

result = con.execute(f"""

WITH journey_with_category AS (

    SELECT

        j.*,
        p.category

    FROM cart_journey j

    LEFT JOIN read_parquet(
        '{DATA_DIR}/product_properties.parquet'
    ) p

        ON j.sku = p.sku
)

SELECT

    category,

    COUNT(*) AS cart_events,

    COUNT(*) FILTER (
        WHERE journey_outcome = 'PURCHASED'
    ) AS purchased,

    COUNT(*) FILTER (
        WHERE journey_outcome = 'REMOVED'
    ) AS removed,

    COUNT(*) FILTER (
        WHERE journey_outcome = 'NO_FOLLOWUP'
    ) AS no_followup,

    ROUND(
        COUNT(*) FILTER (
            WHERE journey_outcome = 'PURCHASED'
        ) * 100.0 / COUNT(*),
        2
    ) AS purchase_rate

FROM journey_with_category

GROUP BY category

HAVING COUNT(*) >= 100

ORDER BY purchase_rate DESC

LIMIT 20

""").fetchdf()

print(result.to_string(index=False))


# ============================================================
# 5. PRODUCTS WITH HIGH CART INTEREST BUT LOW PURCHASE
# ============================================================

print("\n" + "=" * 80)
print("HIGH INTEREST / LOW CONVERSION PRODUCTS")
print("=" * 80)

result = con.execute("""

SELECT

    sku,

    COUNT(*) AS cart_events,

    COUNT(*) FILTER (
        WHERE journey_outcome = 'PURCHASED'
    ) AS purchases_after_cart,

    COUNT(*) FILTER (
        WHERE journey_outcome = 'REMOVED'
    ) AS removals_after_cart,

    COUNT(*) FILTER (
        WHERE journey_outcome = 'NO_FOLLOWUP'
    ) AS no_followup,

    ROUND(
        COUNT(*) FILTER (
            WHERE journey_outcome = 'PURCHASED'
        ) * 100.0 / COUNT(*),
        2
    ) AS purchase_rate

FROM cart_journey

GROUP BY sku

HAVING COUNT(*) >= 20

ORDER BY purchase_rate ASC

LIMIT 20

""").fetchdf()

print(result.to_string(index=False))


# ============================================================
# 6. SAVE A SMALL ANALYTICAL SAMPLE
# ============================================================

print("\n" + "=" * 80)
print("SAMPLE CUSTOMER JOURNEYS")
print("=" * 80)

result = con.execute("""
SELECT

    client_id,
    sku,
    add_time,
    purchase_time,
    remove_time,
    journey_outcome,
    ROUND(seconds_to_outcome / 60, 2) AS minutes_to_outcome

FROM cart_journey

WHERE journey_outcome != 'NO_FOLLOWUP'

ORDER BY add_time

LIMIT 30

""").fetchdf()

print(result.to_string(index=False))

con.close()

print("\nDONE!")

CUSTOMER CART JOURNEY ANALYSIS

Cart journey table created successfully!

CART JOURNEY OUTCOMES
journey_outcome  cart_events  percentage
    NO_FOLLOWUP      3416143       46.29
        REMOVED      2722685       36.89
      PURCHASED      1240796       16.81

TIME TO CART OUTCOME
journey_outcome  events  avg_minutes  median_minutes  min_minutes  max_minutes
        REMOVED 2722685      7966.57            9.08         0.02    240621.33
      PURCHASED 1240796       995.75            5.58         0.02    239174.75

JOURNEY OUTCOME BY CATEGORY
 category  cart_events  purchased  removed  no_followup  purchase_rate
      271          167         66       31           70          39.52
     5143          333        128       66          139          38.44
      620          108         36       38           34          33.33
      813          259         85      101           73          32.82
     5584          205         67       53           85          32.68
     5163          114    